# 06 - Foundation model: Chronos-2 zero-shot

**This notebook is self-contained** so it can run on Google Colab (recommended: GPU runtime, but CPU works too, ~5 min).

Chronos-2 (Ansari et al., 2024) is a pretrained time-series foundation model used here **zero-shot and target-only**: it sees only the history of `Appliances`, no covariates, and is never trained on our data. Forecasts are issued rolling-origin, 24 h at a time, over the final 14 days - the same design as every other model.

After running, download `fc_foundation_model.csv` and `foundation_model_interval.csv` into `outputs/forecasts/` of the repository, then run `python scripts/run_pipeline.py --stage evaluate`.

In [ ]:
%pip install -q chronos-forecasting torch pandas matplotlib

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from chronos import Chronos2Pipeline

## Load and prepare the data (same as the pipeline)

In [ ]:
URL = ('https://archive.ics.uci.edu/ml/machine-learning-databases/00374/energydata_complete.csv')
df = pd.read_csv(URL)
df['date'] = pd.to_datetime(df['date'])
df = df.set_index('date').sort_index()
y = df['Appliances'].resample('h').mean().interpolate('time').dropna()
TEST_STEPS, HORIZON = 14 * 24, 24
train, test = y.iloc[:-TEST_STEPS], y.iloc[-TEST_STEPS:]
print(train.index.min(), '->', test.index.max(), len(y))

## Load Chronos-2

In [ ]:
device_map = 'cuda' if torch.cuda.is_available() else 'cpu'
pipeline = Chronos2Pipeline.from_pretrained('amazon/chronos-2',
                                            device_map=device_map)

## Rolling-origin 24 h forecasts over the test period

One forecast per day; after each day the context is extended with the actual observations (as an operational system would).

In [ ]:
def get_quantile_column(frame, q):
    for col in [q, str(q), f'{q:.1f}', f'{q:.2f}']:
        if col in frame.columns:
            return col
    raise KeyError(f'quantile {q} not in {list(frame.columns)}')

med_all, lo_all, hi_all = [], [], []
for start in range(0, TEST_STEPS, HORIZON):
    window = test.index[start:start + HORIZON]
    history = y.loc[:window[0]].iloc[:-1]
    context_df = pd.DataFrame({'id': 'appliances',
                               'timestamp': history.index,
                               'target': history.to_numpy()})
    pred = pipeline.predict_df(context_df,
                               prediction_length=HORIZON,
                               quantile_levels=[0.1, 0.5, 0.9],
                               id_column='id',
                               timestamp_column='timestamp',
                               target='target')
    pred = pred.sort_values('timestamp').tail(HORIZON)
    med_all.append(pd.Series(
        pred[get_quantile_column(pred, 0.5)].to_numpy(), index=window))
    lo_all.append(pd.Series(
        pred[get_quantile_column(pred, 0.1)].to_numpy(), index=window))
    hi_all.append(pd.Series(
        pred[get_quantile_column(pred, 0.9)].to_numpy(), index=window))
    print('forecast day', start // 24 + 1, 'done')

median = pd.concat(med_all).rename('foundation_model')
lower = pd.concat(lo_all)
upper = pd.concat(hi_all)

## Evaluate and save

In [ ]:
def mae(a, b):
    return float(np.mean(np.abs(np.asarray(a) - np.asarray(b))))
def rmse(a, b):
    return float(np.sqrt(np.mean((np.asarray(a) - np.asarray(b))**2)))
scale = np.mean(np.abs(train.values[24:] - train.values[:-24]))
coverage = float(np.mean((test >= lower) & (test <= upper)))
print(f'MAE  {mae(test, median):.2f}  RMSE {rmse(test, median):.2f}')
print(f'MASE {mae(test, median)/scale:.2f}')
print(f'80% interval coverage: {coverage:.2%}, mean width: '
      f'{float(np.mean(upper - lower)):.1f} Wh')

In [ ]:
median.to_csv('fc_foundation_model.csv')
pd.DataFrame({'lower': lower, 'upper': upper}).to_csv(
    'foundation_model_interval.csv')
print('saved - move both CSVs into outputs/forecasts/')

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(train.tail(72).index, train.tail(72), color='grey',
        label='Training data')
ax.plot(test.index, test, color='black', lw=1.4, label='Actual')
ax.plot(median.index, median, color='tab:red', label='Chronos-2 median')
ax.fill_between(test.index, lower, upper, alpha=0.2,
                color='tab:red', label='Chronos-2 10-90%')
ax.set_ylabel('Appliance energy use (Wh)')
ax.legend()
plt.tight_layout()
plt.savefig('fig12_chronos.png', dpi=150)
plt.show()